# 02 — Prétraitement & DataLoaders
> Définition des transformations, chargement des splits, vérification visuelle.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

from config import TRAIN_DIR, VAL_DIR, TEST_DIR, BATCH_SIZE, IMG_SIZE, MEAN, STD, SEED
from src.dataset import get_train_transform, get_val_transform, get_dataloaders, compute_class_weights

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"PyTorch {torch.__version__} | CUDA : {torch.cuda.is_available()}")

## 1. Transformations définies

In [ ]:
train_tf = get_train_transform()
val_tf   = get_val_transform()

print("=== Train transform (avec augmentation) ===")
for t in train_tf.transforms:
    print(f"  {t}")

print("\n=== Val/Test transform (sans augmentation) ===")
for t in val_tf.transforms:
    print(f"  {t}")

print(f"\n→ Resize cible  : {IMG_SIZE}x{IMG_SIZE}")
print(f"→ Normalisation  : mean={MEAN}, std={STD}")

## 2. Chargement des DataLoaders

In [ ]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=0,   # 0 pour Jupyter, 2+ en production
    use_weighted_sampler=True,
)

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=get_train_transform())
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=get_val_transform())
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=get_val_transform())

print(f"\nClasses : {class_names}  (index : {train_ds.class_to_idx})")
print(f"\nTailles des splits :")
print(f"  Train : {len(train_ds):>5} images  ({len(train_loader)} batches)")
print(f"  Val   : {len(val_ds):>5} images  ({len(val_loader)} batches)")
print(f"  Test  : {len(test_ds):>5} images  ({len(test_loader)} batches)")

## 3. Vérification d'un batch

In [ ]:
images, labels = next(iter(train_loader))
print(f"Shape batch images : {images.shape}   → [batch, canaux, H, W]")
print(f"Shape labels       : {labels.shape}")
print(f"Types labels : {[class_names[l] for l in labels[:8].tolist()]}")
print(f"\nValeurs min/max après normalisation : {images.min():.2f} / {images.max():.2f}")
print("(valeurs négatives normales après normalisation ImageNet)")

## 4. Visualisation après augmentation

In [ ]:
# ✅ CORRIGÉ : bug docstring supprimé (backslash parasite retiré)
def denorm(tensor):
    """Dénormalise un tenseur pour affichage."""
    mean = np.array(MEAN)
    std  = np.array(STD)
    img  = tensor.permute(1, 2, 0).numpy()
    return (img * std + mean).clip(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    ax.imshow(denorm(images[i]))
    ax.set_title(
        class_names[labels[i].item()],
        fontsize=10,
        color='#A32D2D' if class_names[labels[i].item()] == 'Malignant' else '#085041'
    )
    ax.axis('off')
plt.suptitle('Batch train après augmentation (dénormalisé)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Poids de classes (pour la Loss)

In [ ]:
class_weights = compute_class_weights(train_ds)
print("Poids de classes calculés :")
for cls, w in zip(class_names, class_weights.tolist()):
    print(f"  {cls:>12} : {w:.4f}")
print("\n→ Ces poids seront passés à CrossEntropyLoss(weight=class_weights)")
print("→ La classe sous-représentée aura un poids plus élevé (pénalité accrue)")

## 6. Conclusion

In [ ]:
print("✓ DataLoaders prêts")
print("✓ Augmentation train configurée")
print("✓ Poids de classes calculés")
print("\n→ Prochaine étape : 03_training.ipynb")